In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Labs"
)

xpt_file = DATA_DIR / "L13_C.xpt"
csv_file = DATA_DIR / "L13_C_corrected.csv"


# ============================================================
# 2. READ XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep an untouched copy for fidelity comparison
source_df = df.copy()


print("=" * 70)
print("L13_C XPT -> CSV")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Variables:", df.columns.tolist())


# ============================================================
# 3. CHECK TINY ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns

tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

total_tiny = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny values:",
    total_tiny
)

if total_tiny > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    raise ValueError(
        "Tiny values found. "
        "Do not automatically change them "
        "until validated against the codebook."
    )


# ============================================================
# 4. FIX ONLY SEQN FORMATTING
# ============================================================

if "SEQN" in df.columns:

    values = df["SEQN"].dropna()

    # SEQN must be whole-number identifier
    if not np.all(
        np.isclose(
            values,
            np.round(values)
        )
    ):
        raise ValueError(
            "SEQN contains unexpected decimal values."
        )

    df["SEQN"] = (
        df["SEQN"]
        .round()
        .astype("Int64")
    )


# ============================================================
# 5. DO NOT MODIFY LAB VALUES
# ============================================================

#
# IMPORTANT:
#
# No:
#
# .round()
# .astype(int)
# recalculation
# unit conversion
# decimal formatting changes
#
# for laboratory variables.
#

LAB_COLUMNS = [
    "LBXTC",
    "LBXHDD",
    "LBDTCSI",
    "LBDHDDSI"
]


print("\n--- SOURCE LAB VALUES ---")

for col in LAB_COLUMNS:

    if col in df.columns:

        print(
            f"{col}: min={df[col].min()}, "
            f"max={df[col].max()}, "
            f"missing={df[col].isna().sum()}"
        )


# ============================================================
# 6. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)

print("\nCSV created:")
print(csv_file)


# ============================================================
# 7. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    float_precision="round_trip",
    low_memory=False
)


# ============================================================
# 8. EXACT FIDELITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("XPT -> CSV FIDELITY CHECK")
print("=" * 70)


for col in LAB_COLUMNS:

    if col not in source_df.columns:
        continue

    xpt_values = source_df[col].to_numpy()

    csv_values = csv_df[col].to_numpy()

    same = np.allclose(
        xpt_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if same:

        print(
            f"PASS: {col} matches XPT exactly."
        )

    else:

        print(
            f"FAIL: {col} changed."
        )

        differences = (
            ~np.isclose(
                xpt_values,
                csv_values,
                rtol=0,
                atol=0,
                equal_nan=True
            )
        )

        print(
            pd.DataFrame({
                "XPT": source_df.loc[
                    differences,
                    col
                ],
                "CSV": csv_df.loc[
                    differences,
                    col
                ]
            }).head(20)
        )


# ============================================================
# 9. SEQN CHECK
# ============================================================

print("\n--- SEQN CHECK ---")

print(
    "Missing:",
    csv_df["SEQN"].isna().sum()
)

print(
    "Duplicates:",
    csv_df["SEQN"].duplicated().sum()
)


# Check actual CSV text for SEQN .0

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    f.readline()   # header

    bad_seqn = 0

    for line in f:

        seqn = line.split(",", 1)[0]

        if seqn.endswith(".0"):
            bad_seqn += 1


print(
    "SEQN values ending in .0:",
    bad_seqn
)


print("\n" + "=" * 70)
print("COMPLETE")
print("=" * 70)

L13_C XPT -> CSV
Rows: 8556
Columns: 5
Variables: ['SEQN', 'LBXTC', 'LBXHDD', 'LBDTCSI', 'LBDHDDSI']

--- TINY VALUE CHECK ---
Tiny values: 0

--- SOURCE LAB VALUES ---
LBXTC: min=68.0, max=704.0, missing=782
LBXHDD: min=19.0, max=154.0, missing=783
LBDTCSI: min=1.76, max=18.21, missing=782
LBDHDDSI: min=0.49, max=3.98, missing=783

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\2003_2004\Labs\L13_C_corrected.csv

XPT -> CSV FIDELITY CHECK
PASS: LBXTC matches XPT exactly.
PASS: LBXHDD matches XPT exactly.
PASS: LBDTCSI matches XPT exactly.
PASS: LBDHDDSI matches XPT exactly.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
SEQN values ending in .0: 0

COMPLETE
